# Results Comparison
The goal of this notebook is to compare the results
of the finetuned model with the previous baseline one.
The outputs obtained from the baseline and fine tuned model are loaded and compared quantitatively and qualitatively.

The questions we are trying to pose an answer to are:

- Did fine-tuning improve performance on the trained languages?
- Did it transfer to not trained languages (Spanish and Portuguese)?
- Did it change the verbosity of the responses?
- Did it improve exact-match behavior on short-answer tasks?
- Are there examples where fine-tuning clearly helped?
- Are there examples where it hurt?

### Imports

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt


### Paths Definition

In [ ]:
OUTPUTS_DIR = Path("/mnt/data/project_artifacts/outputs")

baseline_test_file = OUTPUTS_DIR / "baseline_test.jsonl"
baseline_spa_file = OUTPUTS_DIR / "baseline_probe_spa.jsonl"
baseline_por_file = OUTPUTS_DIR / "baseline_probe_por.jsonl"

finetuned_test_file = OUTPUTS_DIR / "finetuned_test.jsonl"
finetuned_spa_file = OUTPUTS_DIR / "finetuned_probe_spa.jsonl"
finetuned_por_file = OUTPUTS_DIR / "finetuned_probe_por.jsonl"


Load json files with pandas and do some sanity checks

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)


In [ ]:
baseline_test_df = load_jsonl(baseline_test_file)
baseline_spa_df = load_jsonl(baseline_spa_file)
baseline_por_df = load_jsonl(baseline_por_file)

finetuned_test_df = load_jsonl(finetuned_test_file)
finetuned_spa_df = load_jsonl(finetuned_spa_file)
finetuned_por_df = load_jsonl(finetuned_por_file)


In [ ]:
print("baseline_test:", baseline_test_df.shape)
print("finetuned_test:", finetuned_test_df.shape)

print("baseline_spa:", baseline_spa_df.shape)
print("finetuned_spa:", finetuned_spa_df.shape)

print("baseline_por:", baseline_por_df.shape)
print("finetuned_por:", finetuned_por_df.shape)


In [ ]:
# Here we expect unique indices == rows
for name, df in {
    "baseline_test": baseline_test_df,
    "finetuned_test": finetuned_test_df,
    "baseline_spa": baseline_spa_df,
    "finetuned_spa": finetuned_spa_df,
    "baseline_por": baseline_por_df,
    "finetuned_por": finetuned_por_df,
}.items():
    print(name, "unique indices:", df["index"].nunique(), "rows:", len(df))


Merge the dataset to compare them directly.

In [ ]:
test_compare = baseline_test_df.merge(
    finetuned_test_df,
    on=["index", "language_code"],
    suffixes=("_base", "_ft"),
)

spa_compare = baseline_spa_df.merge(
    finetuned_spa_df,
    on=["index", "language_code"],
    suffixes=("_base", "_ft"),
)

por_compare = baseline_por_df.merge(
    finetuned_por_df,
    on=["index", "language_code"],
    suffixes=("_base", "_ft"),
)


### Evaluation

In this section we define the functions and the ideas after which we will evaluate the results and understand the correctness.

It is important to define what actually means to have good results, since it is not trivial nor absolute. 

1. The first idea is to normalize the text and verify where we get exact matches. This works for short answers or yes/no answers.


In [ ]:
def normalize_text(text):
    return text.strip().lower() # removes spaces at beginning/end and switch it to lowercase


NameError: name 'test_compare' is not defined

In [ ]:
for df in [test_compare, spa_compare, por_compare]:
    df["target_base_norm"] = df["target_base"].apply(normalize_text)
    df["pred_base_norm"] = df["prediction_base"].apply(normalize_text)
    df["pred_ft_norm"] = df["prediction_ft"].apply(normalize_text)

    df["base_exact"] = df["pred_base_norm"] == df["target_base_norm"]
    df["ft_exact"] = df["pred_ft_norm"] == df["target_base_norm"]


2. A second aspect we can use to measure the strength of the model answer is the answer length. The idea is to study whether the model is overexplaining, and if finetuning made the answers shorter or longer.

In [ ]:
def word_count(text):
    return len(text.split())


In [ ]:
for df in [test_compare, spa_compare, por_compare]:
    df["target_len"] = df["target_base"].apply(word_count)
    df["pred_len_base"] = df["prediction_base"].apply(word_count)
    df["pred_len_ft"] = df["prediction_ft"].apply(word_count)


3. Ultimately we simply manually analize a subset of the results and label them. We will mark whether we got a better result for baseline, for the finetuned, if there is a tie or if both are bad.

In [ ]:
manual_test = test_compare.sample(min(20, len(test_compare)), random_state=42).copy()
manual_spa = spa_compare.sample(min(20, len(spa_compare)), random_state=42).copy()
manual_por = por_compare.sample(min(20, len(por_compare)), random_state=42).copy()

for df in [manual_test, manual_spa, manual_por]:
    df["manual_base_score"] = ""
    df["manual_ft_score"] = ""
    df["manual_winner"] = ""
    df["manual_notes"] = ""

manual_test_view = manual_test[[
    "index",
    "language_code",
    "input_base",
    "target_base",
    "prediction_base",
    "prediction_ft",
    "manual_base_score",
    "manual_ft_score",
    "manual_winner",
    "manual_notes",
]]

manual_spa_view = manual_spa[[
    "index",
    "language_code",
    "input_base",
    "target_base",
    "prediction_base",
    "prediction_ft",
    "manual_base_score",
    "manual_ft_score",
    "manual_winner",
    "manual_notes",
]]

manual_por_view = manual_por[[
    "index",
    "language_code",
    "input_base",
    "target_base",
    "prediction_base",
    "prediction_ft",
    "manual_base_score",
    "manual_ft_score",
    "manual_winner",
    "manual_notes",
]]


The scoring system is numeric with:
- 2 correct/strong
- 1 partially correct
- 0 wrong poor

In [ ]:
manual_test_view


In [ ]:
manual_spa_view


In [ ]:
manual_por_view


### Summary
In the following we compute summary tables to analize the results and show them with plots.

In [ ]:
def summarize_exact(df, name):
    summary = {
        "dataset": name,
        "n_examples": len(df),
        "baseline_exact_mean": df["base_exact"].mean(),  # True=1, False=0, so the mean is the exact-match accuracy
        "finetuned_exact_mean": df["ft_exact"].mean(),
        "avg_target_len": df["target_len"].mean(),
        "avg_pred_len_base": df["pred_len_base"].mean(),
        "avg_pred_len_ft": df["pred_len_ft"].mean(),
    }
    return pd.DataFrame([summary])

test_summary = summarize_exact(test_compare, "test")
spa_summary = summarize_exact(spa_compare, "probe_spa")
por_summary = summarize_exact(por_compare, "probe_por")


In [ ]:
test_summary


In [ ]:
spa_summary
# por_summary


While the previous table showed general results, the following only concentrates on the results that improved or worsened the short answers results.

In [ ]:
improved = test_compare[
    (test_compare["base_exact"] == False) &
    (test_compare["ft_exact"] == True)
]

print(improved.length())
improved[["index", "language_code", "input_base", "target_base", "prediction_base", "prediction_ft"]].head(5)


In [ ]:
worsened = test_compare[
    (test_compare["base_exact"] == True) &
    (test_compare["ft_exact"] == False)
]
print(worsened.length())
worsened[["index", "language_code", "input_base", "target_base", "prediction_base", "prediction_ft"]].head(5)


### Plots

In [ ]:
# Exact match performance, one plot for each dataset

for summary_df, title in [
    (test_summary, "Test set"),
    (spa_summary, "Spanish probe set"),
    (por_summary, "Portuguese probe set"),
]:
    ax = summary_df.plot(
        x="dataset",
        y=["baseline_exact_mean", "finetuned_exact_mean"],
        kind="bar",
        rot=0,
        figsize=(5, 4),
        legend=True,
    )

    ax.set_ylabel("Exact match rate")
    ax.set_xlabel("")
    ax.set_title(f"Baseline vs Fine-tuned — {title}")
    ax.set_ylim(0, 1)
    ax.legend(["Baseline", "Fine-tuned"])
    plt.tight_layout()
    plt.show()


In [ ]:
# Plotting the Average description length

for summary_df, title in [
    (test_summary, "Test set"),
    (spa_summary, "Spanish probe set"),
    (por_summary, "Portuguese probe set"),
]:
    ax = summary_df.plot(
        x="dataset",
        y=["avg_target_len", "avg_pred_len_base", "avg_pred_len_ft"],
        kind="bar",
        rot=0,
        figsize=(5, 4),
        legend=True,
    )

    plt.ylabel("Average answer length (words)")
    ax.set_xlabel("")
    ax.set_title(f"Target vs Baseline vs Fine-tuned Answer Length — {title}")
    ax.set_ylim(0, 1)
    ax.legend(["Baseline", "Fine-tuned"])
    plt.tight_layout()
    plt.show()


# Conclusions

### Non Language performance

The study could be continued by analyzing the performance of the model baseline/fine-tuned on aspects which are uncorrelated to the language skills of the model.

Since fine tuning can be interpreted as an "attribute swithching" it can be interested to see whether becoming more intelligent semantically on different languages has impacted the ability of the model on other uncorrelated tasks.